In [1]:
# Web scraping dan automatisasi
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
# from selenium.webdriver.common.action_chains import ActionChains

# Parsing dan data handling
from bs4 import BeautifulSoup
import pandas as pd
import requests
import json
import re
import os
import time
import warnings

In [3]:
from pandas import options


class RegencyCityScraper:
    def __init__(self):
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        self.options = Options()
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_argument('--start-maximized')
        
        # Force UI language
        self.options.add_argument("--lang=en-US")

        # Force Accept-Language header
        self.options.add_experimental_option(
            "prefs",
            {"intl.accept_languages": "en,en_US"}
        )

        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        self.driver = webdriver.Chrome(options=self.options)
        self.wait = WebDriverWait(self.driver, 10)

    def scrollPage(self):
        """ Scroll the results paneluntil no more results of City/Region in Indonesia"""
        
        print("Scrolling to load all results...")

        try:

            table = self.wait.until(
                EC.presence_of_element_located((By.ID, "mw2g"))
            )

            self.driver.execute_script(
                "arguments[0].scrollIntoView({block: 'start'});",
                table
            )
            time.sleep(1.5)

            print("Table found. Scrolling table content...")
            print("-" * 50)

            previousCount = 0
            scrollIterations = 0

            # Infinite scroll until no new results are loaded
            while True:
                rows = table.find_elements(By.TAG_NAME, 'tr')[1:]
                currentCount = len(rows)
                print(f"Current loaded results: {currentCount}")

                self.driver.execute_script(
                    "arguments[0].scrollTop = arguments[0].scrollHeight",
                    table
                )
                time.sleep(1.5)
                scrollIterations += 1

                print(f"Scroll {scrollIterations}. Found {currentCount} results loaded.")
                
                if currentCount == previousCount:
                    print("No new results loaded. Finished scrolling.")
                    break

                previousCount = currentCount
        except NoSuchElementException as e:
            print(f"Not find scrollable element: {e}")

    def mainScraping(self, url):
        """ Extract City/Region data from the loaded page """
        self.driver.get(url)
        time.sleep(4)

        self.scrollPage()
        time.sleep(4)
        
        data = []
        
        rows = self.driver.find_elements(By.XPATH, '//tbody[@id="mw2g"]/tr')[1:]
        print("Total rows:", len(rows))

        print("-" * 14 + " Start Extracting Data " + "-" * 14)
        # Looping through each row to extract data
        for i, row in enumerate(rows, 1):
            try:
                column = row.find_elements(By.XPATH, './/td')
                
                if len(column) >= 5:
                    # Extract Regency Name
                    try:
                        regencyCityElement = column[1].find_element(By.TAG_NAME, 'a')
                        regencyCityTitle = regencyCityElement.get_attribute('title')
                    except:
                        regencyCityTitle = column[1].text.strip()

                    # Extract Province Name
                    try:
                        provinceElement = column[4].find_element(By.TAG_NAME, 'a')
                        provinceTitle = provinceElement.get_attribute('title')
                    except:
                        provinceTitle = column[4].text.strip()

                    print(f"[{i}/{len(rows)}] Get Data Province: {provinceTitle} | Regency/City: {regencyCityTitle}")

                    # Append to data list
                    data.append({
                        "Regency/City": regencyCityTitle,
                        "Province": provinceTitle
                    })

            except NoSuchElementException:
                continue
        return pd.DataFrame(data)

    def mappingProvinceToIsland(self, df):
        """
        Added Island column based on Province mapping
        1. Sumatera
        2. Jawa
        3. Kalimantan
        4. Sulawesi
        5. Nusa Tenggara & Bali
        6. Papua & Maluku
        7. Lainnya
        """

        province_to_island = {
            # Sumatera
            "Aceh": "Sumatera",
            "Sumatera Utara": "Sumatera",
            "Sumatera Barat": "Sumatera",
            "Riau": "Sumatera",
            "Jambi": "Sumatera",
            "Sumatera Selatan": "Sumatera",
            "Bengkulu": "Sumatera",
            "Lampung": "Sumatera",
            "Kepulauan Bangka Belitung": "Sumatera",
            "Kepulauan Riau": "Sumatera",

            # Jawa
            "Daerah Khusus Ibukota Jakarta": "Jawa",
            "Banten": "Jawa",
            "Jawa Barat": "Jawa",
            "Jawa Tengah": "Jawa",
            "Jawa Timur": "Jawa",
            "Daerah Istimewa Yogyakarta": "Jawa",

            # Kalimantan
            "Kalimantan Barat": "Kalimantan",
            "Kalimantan Tengah": "Kalimantan",
            "Kalimantan Selatan": "Kalimantan",
            "Kalimantan Timur": "Kalimantan",
            "Kalimantan Utara": "Kalimantan",

            # Sulawesi
            "Sulawesi Utara": "Sulawesi",
            "Sulawesi Tengah": "Sulawesi",
            "Sulawesi Selatan": "Sulawesi",
            "Sulawesi Tenggara": "Sulawesi",
            "Sulawesi Barat": "Sulawesi",
            "Gorontalo": "Sulawesi",

            # Bali & Nusa Tenggara
            "Bali": "Nusa Tenggara & Bali",
            "Nusa Tenggara Barat": "Nusa Tenggara & Bali",
            "Nusa Tenggara Timur": "Nusa Tenggara & Bali",

            # Papua & Maluku
            "Maluku": "Papua & Maluku",
            "Maluku Utara": "Papua & Maluku",
            "Papua": "Papua & Maluku",
            "Papua Barat": "Papua & Maluku",
            "Papua Barat Daya": "Papua & Maluku",
            "Papua Tengah": "Papua & Maluku",
            "Papua Pegunungan": "Papua & Maluku",
            "Papua Selatan": "Papua & Maluku",
        }

        df = df.copy()
        df["Island"] = df["Province"].map(province_to_island).fillna("Lainnya")
        return df
    
    def save_to_excel(self, df, filename):
        """ Save DataFrame to Excel file """
        df.to_excel(filename, index=False)
        print("-" * 50)
        print(f"Saved to {filename}")
        print(f"Total Regency/City: {df['Regency/City'].nunique()}")
    
    def close(self):
        """ Close the browser """
        self.driver.quit()

if __name__ == "__main__":
    scraper = RegencyCityScraper()
    url = "https://id.wikipedia.org/wiki/Kabupaten_dan_kota_di_Indonesia"
    df = scraper.mainScraping(url)
    df = scraper.mappingProvinceToIsland(df)
    scraper.save_to_excel(df, "kota_indonesia.xlsx")
    scraper.close()

Scrolling to load all results...
Table found. Scrolling table content...
--------------------------------------------------
Current loaded results: 513
Scroll 1. Found 513 results loaded.
Current loaded results: 513
Scroll 2. Found 513 results loaded.
No new results loaded. Finished scrolling.
Total rows: 513
-------------- Start Extracting Data --------------
[1/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Barat Daya
[2/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Besar
[3/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Jaya
[4/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Selatan
[5/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Singkil
[6/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tamiang
[7/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tengah
[8/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tenggara
[9/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Timur
[1

In [ ]:
class scraperGoogleMaps:
    """ Class to scrape Google Maps store data """
    def __init__(self):
        """" Initialize the scraper with Chrome options """
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        self.options = Options()
        self.options.add_argument("--lang=en-US")  # Force browser language to English
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_argument('--start-maximized')

        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)

        # Force browser to use English language
        self.options.add_experimental_option(
            "prefs",
            {"intl.accept_languages": "en,en_US"}
        )

        self.driver = webdriver.Chrome(options=self.options)
        self.wait = WebDriverWait(self.driver, 10)
        self.time_sleep = 10  # fixed, in seconds

    def scroll_feed(self):
        """ Scroll the results panel until no more results are loaded """

        print("Scrolling to load all results...")
        print("-" * 50)

        try:
            # Find the scrollable element
            scrollable_element = self.wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, '//div[@role="feed"]')
                )
            )

            previous_store_count = 0
            iteration = 0

            # Infinite scroll until no new results are loaded
            while True:

                store_article = self.driver.find_elements(
                    By.XPATH, './/div[@role="article"]'
                )
                store_count = len(store_article)

                print(f"Current loaded results: {store_count}")

                self.driver.execute_script(
                    "arguments[0].scrollTop = arguments[0].scrollHeight",
                    scrollable_element
                )

                time.sleep(1.5)
                iteration += 1

                print(f"Scroll {iteration}. Found {store_count} results loaded.")

                if store_count == previous_store_count:

                    print("No new results loaded. Finished scrolling.")
                    break

                previous_store_count = store_count

        except NoSuchElementException as e:
            print(f"Not find scrollable element: {e}")
    
    def load_checkpoint(self, checkpoint_file):
        """ Load checkpoint data from file """

        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, "r", encoding="utf-8") as f:
                result = json.load(f)
                print("Checkpoint loaded. Total data sebelumnya:", len(result))

            set_data_checks = set()
            for item in result:
                key = f"{item.get('Name')}|{item.get('Latitude')}|{item.get('Longitude')}"
                set_data_checks.add(key)
        else:
            os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)
            print("No checkpoint found. Starting fresh scrape.")
            result = []
            set_data_checks = set()
        
        return result, set_data_checks
    
    def save_checkpoint(self, checkpoint_file, data):
        """ Save checkpoint data to file """
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)

    def extract_store_links(self):
        """ Extract store links from the loaded page """
        
        store_links = []

        # Find all store articles
        store_articles = self.wait.until(
            EC.presence_of_all_elements_located(
                (By.XPATH, './/div[@role="article"]')
            )
        )

        print("Total stores found:", len(store_articles))
        print("-" * 9 + " Start Extracting Store Links " + "-" * 9)

        # Loop through each store article to extract links
        for i, article in enumerate(store_articles, 1):

            try:

                link_element = article.find_element(
                    By.XPATH, './/a[contains(@href, "/place/")]'
                )
                store_link = link_element.get_attribute('href')

                if store_link not in store_links:
                    store_links.append(store_link)

                print(f"✅ [{i}/{len(store_articles)}] Successfully extracted store link")

            except NoSuchElementException:
                print(f"❌ [{i}/{len(store_articles)}] Failed to extract store link")
                continue
        return store_links

    @staticmethod
    def extract_lat_long(url):
        """ 
        Extract latitude and longitude from the URL

        arguments:
        - url : str
            The URL containing latitude and longitude information.
        
        """
        lat, long = "", ""

        try:
            # Use regex to find lat and long in the URL
            match = re.search(r'(@|!3d)([-.\d]+)[,!4d]+([-.\d]+)', url)

            # If match is found, extract lat and long
            if match:
                lat = match.group(2)
                long = match.group(3)

        except Exception as e:
            print(f"Error extracting lat/long: {e}")
        return lat, long

    def extract_status(self):
        """ Extract open/closed status from the current page """

        status = ""

        try:
            # Extract status using different conditions
            ## Condition 1: Check hours element to determine status using JS
            status = self.driver.execute_script("""
                const el = document.querySelector('[aria-label="Hours"]')?.parentElement;
                return el ? el.innerText.trim() : "";
            """)

            ## Condition 2: Check notice element to determine status using JS
            if not status:
                status = self.driver.execute_script("""
                    const el = document.querySelector('[aria-label="Notice"]')?.parentElement;
                    return el ? el.innerText.trim() : "";
                """)

            # Normalize status text
            ## Case 1 - Permanently Closed / Temporarily Closed
            if "permanently closed" in status.lower():
                return "Permanently Closed"

            if "temporarily closed" in status.lower():
                return "Temporarily Closed"

            ## Case 2 - Open / Closed based on hours
            if "open" in status.lower() and "closes" in status.lower():
                return "Open"

            if "closed" in status.lower() and "opens" in status.lower():
                return "Closed"
            
            if "opens" in status.lower():
                return "Open"
            
            if "closed" in status.lower():
                return "Closed"

        except Exception:
            return ""
        
        return status

    def extract_share_link(self):
        """ Extract share link from the current page """

        try:
            # Click Share button
            share_btn = self.wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//button[contains(@aria-label,"Share") or contains(@aria-label,"Bagikan")]')
                )
            )
            self.driver.execute_script("arguments[0].click();", share_btn)
            time.sleep(3)

            share_link = ""

            # Condition 1:Try old input UI
            try:
                input_el = WebDriverWait(self.driver, 3).until(
                    EC.presence_of_element_located(
                        (By.XPATH, '//input[@readonly and contains(@value,"maps.app.goo.gl")]')
                    )
                )
                share_link = input_el.get_attribute("value")

            except TimeoutException:

                # Condition 2: Try new span UI
                try:
                    span_el = WebDriverWait(self.driver, 3).until(
                        EC.presence_of_element_located(
                            (By.XPATH, '//span[contains(@class,"htP7Y") and contains(text(),"maps.app.goo.gl")]')
                        )
                    )
                    share_link = span_el.text.strip()

                except TimeoutException:
                    share_link = self.driver.current_url

            # Close Pop Up
            body = self.driver.find_element(By.TAG_NAME, "body")
            body.send_keys(Keys.ESCAPE)
            time.sleep(0.5)

            return share_link

        except Exception as e:
            return self.driver.current_url
        
    @staticmethod
    def normalize_city_regency(regency):
        """
        Normalize city/regency naming variations into a single standard format
        """

        # Check if regency is valid
        if not regency:
            return None

        # Basic Preprocess input
        regency = regency.lower()
        regency = re.sub(r"[^\w\s]", " ", regency)
        regency = re.sub(r"\s+", " ", regency).strip()

        # Mapping cardinal directions
        cardinal_map = {
            r"\bnorth\s*[-]?\s*east\b": "timur laut",
            r"\bnorth\s*[-]?\s*west\b": "barat laut",
            r"\bsouth\s*[-]?\s*east\b": "tenggara",
            r"\bsouth\s*[-]?\s*west\b": "barat daya",
            r"\bnorth\b": "utara",
            r"\bsouth\b": "selatan",
            r"\beast\b": "timur",
            r"\bwest\b": "barat",
            r"\bcentral\b": "tengah",
        }

        # Apply cardinal direction mappings
        for pattern, repl in cardinal_map.items():
            regency = re.sub(pattern, repl, regency, flags=re.IGNORECASE)

        # Identify and remove administration labels
        adm_label = ""

        if re.search(r"\b(kota|city)\b", regency, flags=re.IGNORECASE):
            adm_label = "Kota"
            regency = re.sub(r"\b(kota|city)\b", "", regency, flags=re.IGNORECASE)

        elif re.search(r"\b(kab|kab\.|kabupaten|regency)\b", regency, flags=re.IGNORECASE):
            adm_label = "Kabupaten"
            regency = re.sub(r"\b(kab|kab\.|kabupaten|regency)\b", "", regency, flags=re.IGNORECASE)

        # Reattach direction to the end if it was at the start
        directions = [
            "timur laut", "barat laut", "tenggara", "barat daya",
            "utara", "selatan", "timur", "barat", "tengah"
        ]

        for d in directions:
            if regency.startswith(d + " "):
                regency = regency.replace(d + " ", "") + " " + d
                break

        # Clean up extra spaces
        regency = re.sub(r"\s+", " ", regency).strip()
        regency = " ".join(word.capitalize() for word in regency.split())

        # Reattach administration label
        if adm_label:
            regency_reattach = f"{adm_label} {regency}"

            return regency_reattach
        
        return regency
    
    @staticmethod
    def normalize_province(province):
        """
        Normalize province naming variations into a single standard format
        """

        if not province:
            return None
        
        # Special Cases: if any Jakarta in Province, change to DKI Jakarta
        if re.search(r"\bjakarta\b", province):
            return "DKI Jakarta"
        
        # Special Cases: if any Yogyakarta in Province, change to Daerah Istimewa Yogyakarta
        if re.search(r"\byogyakarta\b", province):
            return "Daerah Istimewa Yogyakarta"

        # Mapping cardinal directions
        cardinal_map = {
            r"\bnorth\s*[-]?\s*east\b": "timur laut",
            r"\bnorth\s*[-]?\s*west\b": "barat laut",
            r"\bsouth\s*[-]?\s*east\b": "tenggara",
            r"\bsouth\s*[-]?\s*west\b": "barat daya",
            r"\bnorth\b": "utara",
            r"\bsouth\b": "selatan",
            r"\beast\b": "timur",
            r"\bwest\b": "barat",
            r"\bcentral\b": "tengah",
        }

        # Apply cardinal direction mappings
        for pattern, repl in cardinal_map.items():
            province = re.sub(pattern, repl, province, flags=re.IGNORECASE)

        # Identify and remove administration labels
        adm_label = ""

        if re.search(r"\b(province|prov.)\b", province, flags=re.IGNORECASE):
            adm_label = "Prov."
            province = re.sub(r"\b(province|prov.)\b", "", province, flags=re.IGNORECASE)

        # Reattach direction to the end if it was at the start
        directions = [
            "timur laut", "barat laut", "tenggara", "barat daya",
            "utara", "selatan", "timur", "barat", "tengah"
        ]

        for d in directions:
            if province.startswith(d + " "):
                province = province.replace(d + " ", "") + " " + d
                break

        # Reattach administration label
        if adm_label:
            province_reattach = f"{adm_label} {province}"

            return province_reattach

        return province
     
    @staticmethod
    def extract_city_province(address):
        """ 
        Extract City/Regency and Province from address using simple parsing
        """

        if not address:
            return None, None

        parts = [part.strip() for part in address.split(',')]

        if len(parts) >= 2:
            province = parts[-1]
            regency = parts[-2]

            # hapus kode pos
            province = re.sub(r"\b\d{5}\b", "", province).strip()
            regency = re.sub(r"\b\d{5}\b", "", regency).strip()

            # Use normalizer
            regency = scraperGoogleMaps.normalize_city_regency(regency)

            # rapikan province (tanpa normalisasi kota/kab)
            province = scraperGoogleMaps.normalize_province(province)

            return province, regency

    def extract_details_store(self, url, file_zip_code):
        """ 
        Extract detailed store information from the current page

        arguments:
        - url : str
            The URL of the store page.
        - file_zip_code : pd.DataFrame
            DataFrame containing zip code data.

        """

        details = {}

        try:
            
            # Extract Store Name
            try:
                name_element = self.wait.until(
                    EC.presence_of_element_located(
                        (By.XPATH, '//h1[contains(@class,"DUwDvf")]')
                    )
                )

                store_name = self.driver.execute_script(
                    "return arguments[0].textContent;", name_element
                ).strip()

                details["Name"] = store_name

            except TimeoutException:
                details['Name'] = ""

            # Extract Address
            try:
                address_button = self.wait.until(
                    EC.presence_of_element_located(
                        (By.XPATH, '//button[@data-item-id="address"]')
                    )
                ).get_attribute("aria-label")

                # Clean address text
                address = address_button.replace("Address:", "").strip()

                details["Address"] = address

            except NoSuchElementException:
                details['Address'] = ""

            # Extract City/Regency and Province
            province, city_regency = self.extract_city_province(
                address
            )
            details['City/Regency'] = city_regency
            details['Province'] = province

            # Extract Status
            details['Status'] = self.extract_status()

            # Extract Latitude and Longitude
            lat, long = self.extract_lat_long(url)
            details['Latitude'] = lat
            details['Longitude'] = long

            # Extract Store Link
            store_link = self.extract_share_link()
            details['Store Link'] = store_link

        except Exception as e:
            print(f"Error extracting store details: {e}")
        return details

    def scrape(self, competitor, island, kota_df, zip_df, max_time):
        """ 
        Main scraping function

        arguments:
        - competitor : str
            The name of the competitor to search for.
        - island : str
            The island to filter cities/regencies.
        - kota_df : pd.DataFrame
            DataFrame containing city/regency data.
        - zip_df : pd.DataFrame
            DataFrame containing zip code data.

        """

        start_time = time.time()  # Start run time
        last_checkpoint_time = start_time  # start checkpoint time
        checkpoint_interval = 60  # Checkpoint in seconds

        need_checkpoint_save = False

        checkpoint_file = os.path.join("checkpoint file", f"checkpoint_{competitor}.json")

        result, set_data_checks = self.load_checkpoint(checkpoint_file)

        completed_cities_list = list(set([item["City/Regency"] for item in result]))

        self.driver.get("https://www.google.com/maps?hl=en")
        time.sleep(5)

        # Filter kota_df based on the specified island
        if island == "None":
            filtered = kota_df
        else:
            filtered = kota_df[kota_df["Island"] == island.title()]

        # Loop through each city/regency to perform search and extract data
        for idx, row in filtered.iterrows():

            current_time = time.time()
            if current_time - start_time >= max_time:
                print("Maximum scraping time reached. Stopping the scraper.")
                break

            # Get city and province
            city = row["Regency/City"]
            province = row["Province"]

            if city in completed_cities_list:
                print(f"\n ✅ {city} has been scraped. Skipping...")
                continue

            # Perform search query
            query = f"{competitor} di dekat {city} Provinsi {province}"
            print(f"\nSearching: {query}")

            # Enter search query
            search_box = self.wait.until(
                EC.presence_of_element_located((By.ID, "searchboxinput"))
            )
            search_box.clear()
            search_box.send_keys(query)
            search_box.send_keys(Keys.ENTER)

            time.sleep(5)

            # Check if multiple results are present
            try:
                feeds = WebDriverWait(self.driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, '//div[@role="feed"]'))
                )
                print("Multiple results detected")
                is_multiple = True

            except TimeoutException:
                print("Single result page detected")
                is_multiple = False

            # Scroll if multiple results found
            if is_multiple:
                self.scroll_feed() 

                # Extract store links from the results
                links = self.extract_store_links()
                
                print("-"*5 + f" Start extracting details for {len(links)} stores " + "-"*5)

                # Loop through each store link to extract details
                for i, link in enumerate(links, 1):

                    current_time = time.time()
                    if current_time - start_time >= max_time:
                        print("Max time reached while extracting store links.")
                        break

                    self.driver.get(link)
                    time.sleep(self.time_sleep)

                    detail = self.extract_details_store(link, zip_df)

                    data_check = f"{detail.get('Name')}|{detail.get('Latitude')}|{detail.get('Longitude')}"

                    if data_check in set_data_checks:
                        print(f"✅ [{i}/{len(links)}] Duplicate detected, skipped.")
                        print("-" * 30)
                        continue

                    result.append(detail)
                    set_data_checks.add(data_check)

                    # Checkpoint delayed save needed
                    if current_time - last_checkpoint_time >= checkpoint_interval:
                        need_checkpoint_save = True

                    print(f"[{i}/{len(links)}] {detail['Name']}")
                    print(f"Store Name: {detail['Name']}")
                    print(f"Address: {detail['Address']}")
                    print(f"City/Regency: {detail['City/Regency']}, Province: {detail['Province']}")
                    print(f"Latitude: {detail['Latitude']}, Longitude: {detail['Longitude']}")
                    print(f"Status: {detail['Status']}")
                    print(f"Store Link: {detail['Store Link']}")
                    print("-" * 30)

                # Save checkpoint after all stores are done
                if need_checkpoint_save:
                    self.save_checkpoint(checkpoint_file, result)
                    print("Checkpoint saved.")
                    last_checkpoint_time = time.time()
                    need_checkpoint_save = False
            else:
                current_time = time.time()
                if current_time - start_time >= max_time:
                    print("Maximum scraping time reached on single store page.")
                    break

                print("Search result is single place, no scrolling needed.")
                print("-" * 50)

                current_url = self.driver.current_url

                detail = self.extract_details_store(current_url, zip_df)

                data_check = f"{detail.get('Name')}|{detail.get('Latitude')}|{detail.get('Longitude')}"

                if data_check in set_data_checks:
                    print("✅ [1/1] Duplicate detected, skipped.")
                    print("-" * 30)
                    continue
                else:
                    result.append(detail)
                    set_data_checks.add(data_check)

                print(f"[1/1] {detail['Name']}")
                print(f"Store Name: {detail['Name']}")
                print(f"Address: {detail['Address']}")
                print(f"City/Regency: {detail['City/Regency']}, Province: {detail['Province']}")
                print(f"Latitude: {detail['Latitude']}, Longitude: {detail['Longitude']}")
                print(f"Status: {detail['Status']}")
                print(f"Store Link: {detail['Store Link']}")
                print("-" * 30)

                # Checkpoint delayed save needed
                if current_time - last_checkpoint_time >= checkpoint_interval:
                    need_checkpoint_save = True

            # Save checkpoint after each city/regency
            if need_checkpoint_save:
                self.save_checkpoint(checkpoint_file, result)
                print("Checkpoint saved.")
                last_checkpoint_time = time.time()
                need_checkpoint_save = False

        # Save final checkpoint
        self.save_checkpoint(checkpoint_file, result)
        print("Final checkpoint saved.")

        return result

    def save_to_excel(self, data, filename, folder_name):
        """ 
        Save scraped data to Excel file 

        arguments:
        data : list of dict
            The scraped data to be saved.
        filename : str
            The name of the output Excel file.
            
        """

        df = pd.DataFrame(data)

        if os.path.exists(folder_name):
            filename = os.path.join(folder_name, filename)
            df.to_excel(filename, index=False, engine='openpyxl')
        else:
            os.makedirs(folder_name)
            filename = os.path.join(folder_name, filename)
            df.to_excel(filename, index=False, engine='openpyxl')

        print("-" * 50)
        print(f"Data saved to {filename}")
        print(f"Total stores scraped: {len(df)}")

    def close(self):
        """ Close the browser """
        self.driver.quit()

if __name__ == "__main__":
    
    # Input Maximum Time Scraping
    hours = int(input("Masukkan waktu maksimum scraping (dalam jam): "))
    minutes = int(input("Masukkan waktu maksimum scraping (dalam menit): "))
    seconds = int(input("Masukkan waktu maksimum scraping (dalam detik): "))
    max_time = hours * 3600 + minutes * 60 + seconds  # Convert to seconds
    print(f"Waktu Maksimum Scraping: {max_time} detik")

    # Input Competitor Name
    competitor = str(input("Masukkan nama kompetitor yang ingin dicari: "))
    print(f"Nama Competitor: {competitor}")

    # Input Island Name
    ## If no island filter is needed, input "None"
    ## check island name in kota_indonesia.xlsx
    island = str(input("Masukkan nama pulau (capitalize) yang ingin discrape (ex: Jawa, Sumatera, etc): "))
    print(f"Nama Island: {island}")

    print("-" * 100)

    # Start Scraping
    scraper = scraperGoogleMaps()
    file_kota = pd.read_excel("kota_indonesia.xlsx", engine='openpyxl')
    file_zip_code = pd.read_excel("zip_code_indonesia.xlsx", engine='openpyxl')
    date_now = time.strftime("%Y%m%d")
    data = scraper.scrape(competitor, island, file_kota, file_zip_code, max_time)
    scraper.save_to_excel(data, f"scraped_location_{competitor}_{date_now}.xlsx", "Data Hasil Scraping")
    scraper.close()

Waktu Maksimum Scraping: 10800 detik
Nama Competitor: miniso
Nama Island: None
----------------------------------------------------------------------------------------------------
Checkpoint loaded. Total data sebelumnya: 4

Searching: miniso di dekat Kabupaten Aceh Barat Daya Provinsi Aceh
Single result page detected
Search result is single place, no scrolling needed.
--------------------------------------------------
✅ [1/1] Duplicate detected, skipped.
------------------------------

Searching: miniso di dekat Kabupaten Aceh Besar Provinsi Aceh
Single result page detected
Search result is single place, no scrolling needed.
--------------------------------------------------
✅ [1/1] Duplicate detected, skipped.
------------------------------

Searching: miniso di dekat Kabupaten Aceh Jaya Provinsi Aceh
Single result page detected
Search result is single place, no scrolling needed.
--------------------------------------------------
✅ [1/1] Duplicate detected, skipped.
-----------------